# MNIST 1 vs 7 — Classification and Clustering

**Student name:** `<Your Full Name>`
**Student ID:** `<1234567>`
**Course:** `<Course code and name>`

---

### How to use this template
- Every `# TODO` tells you what to implement in that cell. Delete the `raise NotImplementedError` line once done.
- `HD tip` comments point to what the rubric rewards at High Distinction level.
- Cells marked **[provided]** are boilerplate that already works — you can leave them as they are.
- Run top to bottom (*Kernel → Restart & Run All*) before submitting so every output and figure is visible.
- Every figure is saved to `figures/` with `save_fig(...)` so you can drop it straight into the report.

### Golden rule
The **test set is never used** for training, scaling, feature selection, hyperparameter tuning or model selection.
It is touched in exactly two places:
1. **Section 3.7** — one final, single evaluation of the two already-tuned models.
2. **Task 4** — clustering the digit-7 test images (the assignment explicitly asks for this).

### Notebook map
| Section | Task | Marks |
|---|---|---|
| 0 | Setup and data loading | — |
| 1 | Data exploration and preprocessing | 15% |
| 2 | Classifier 1 (Logistic Regression) and metrics | 25% |
| 3 | Classifier 2 (RBF SVM), cross-validation, tuning, comparison | 35% |
| 4 | K-means clustering of test-set 7s | 25% |

## 0. Setup

In [ ]:
# [provided] Imports
# Keep all imports in one place so a marker can see every dependency at a glance.
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV,
    cross_validate, learning_curve,
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, pairwise_distances_argmin_min,
)
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# [provided] Global configuration
RANDOM_STATE = 42          # one seed, used everywhere -> fully reproducible results
DATA_PATH = "mnist_1_7_subsets.npz"   # put the LMS file next to this notebook
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

# Black-and-white plotting style: series are told apart by line style and marker,
# never by colour, so every figure prints cleanly in greyscale.
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.grid": True,
    "grid.color": "0.85",
    "grid.linestyle": ":",
    "image.cmap": "gray_r",   # digits drawn black on white, like the original scans
    "axes.prop_cycle": (cycler(color=["black", "0.35", "0.6"])
                        * cycler(linestyle=["-", "--", ":"])
                        + cycler(marker=["o", "s", "^", "D", "v", "x", "P", "*", "<"]) ),
})


def save_fig(fig, name):
    # Save a figure as PNG into figures/ so it can be inserted into the report.
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path)
    print(f"saved -> {path}")

In [ ]:
# [provided] Data loading
def load_data(path=DATA_PATH):
    # Load the four arrays from the course .npz file.
    # Returns: X_train (100,28,28), y_train (100,), X_test (1000,28,28), y_test (1000,)
    data = np.load(path)
    print("Arrays in file:", data.files)
    return (data["train_data"], data["train_targets"],
            data["test_data"], data["test_targets"])


X_train_img, y_train_raw, X_test_img, y_test_raw = load_data()

# The test arrays are loaded here only so the file is read once.
# Do NOT look at them again until Section 3.7 and Task 4.

---
# Task 1 — Data Exploration and Analysis (15%)

**What the rubric rewards at HD**
1. **≥3 statistics** *and* analysis that **links each finding to the classification task** (not just describing numbers).
2. **≥2 well-labelled visualisations**, each with axis labels, a title, a caption in the report, and a one-line interpretation.
3. Preprocessing steps that are **justified by what you found** in the data.

Use the **training set only** in this section.

### 1.1 Basic inspection and summary statistics

In [ ]:
def inspect_dataset(X_img, y):
    # TODO: Print/return a summary table (pandas DataFrame) with at least these rows:
    #   - number of samples, image shape, flattened feature count (784)
    #   - dtype, min and max pixel value (confirms the 0-255 range)
    #   - count AND percentage of each class (1 and 7) -> is the set balanced?
    #   - % of pixels that are exactly 0 (sparsity)
    #   - number of pixels that are 0 in EVERY training image (zero-variance features)
    #   - any NaN / missing values
    # Hint: np.unique(y, return_counts=True), (X_img == 0).mean(), X_img.reshape(len(X_img), -1).std(axis=0)
    # HD tip: next to each statistic, add a short "Implication" column, e.g.
    #   "balanced 5x/5y classes -> accuracy is a fair headline metric"
    #   "~N pixels never used -> they carry no information; harmless for regularised models,
    #    but would be divided by zero by a naive standardiser"
    raise NotImplementedError


summary_df = inspect_dataset(X_train_img, y_train_raw)
summary_df

In [ ]:
def per_class_statistics(X_img, y):
    # TODO: Compute per-class descriptive statistics and return a DataFrame, e.g.
    #   - "ink" per image = number of non-zero pixels (or sum of intensities / 255)
    #   - mean +- std of ink per class
    #   - bounding-box width and height of the stroke per class
    #     (hint: np.where(img > 0) gives row/col coords; width = max_col - min_col + 1)
    #   - mean intensity of non-zero pixels per class
    # HD tip: these are hand-crafted features. If 7s have clearly more ink and wider
    # bounding boxes than 1s, say so: it tells you the classes are easy to separate
    # and that even a simple (linear) model should do well.
    raise NotImplementedError


class_stats_df = per_class_statistics(X_train_img, y_train_raw)
class_stats_df

### 1.2 Visualisations
Aim for **4 figures**. Each must have a title, axis labels or tick labels where relevant, and you must state what it shows in the report.

In [ ]:
def plot_sample_grid(X_img, y, n_per_class=10):
    # TODO: Figure 1 - a 2-row grid of example images (row 1 = digit 1, row 2 = digit 7).
    #   - Use imshow(img, cmap="gray_r"), turn axes off, label each row with its class.
    #   - Pick samples with a fixed RNG so the figure is reproducible.
    # HD tip: comment on the within-class variety you see (slant, stroke thickness,
    # 7s with/without a crossbar, 1s with a serif/base). Some 1s with a top hook look like 7s
    # -> these are the likely hard cases for the classifier.
    raise NotImplementedError


fig = plot_sample_grid(X_train_img, y_train_raw)
save_fig(fig, "fig1_sample_grid")

In [ ]:
def plot_class_means(X_img, y):
    # TODO: Figure 2 - three panels side by side:
    #   (a) mean image of digit 1, (b) mean image of digit 7,
    #   (c) difference image (mean_7 - mean_1) with a diverging greyscale map and a colorbar.
    # Hint: X_img[y == 7].mean(axis=0); for (c) use cmap="gray" with vmin=-m, vmax=+m (symmetric).
    # HD tip: panel (c) is a picture of where the discriminative information lives
    # (the top horizontal bar of the 7). Point out that a linear model learns weights
    # that look much like this map -> a direct link to your model choice in Task 2.
    raise NotImplementedError


fig = plot_class_means(X_train_img, y_train_raw)
save_fig(fig, "fig2_class_means")

In [ ]:
def plot_feature_distributions(X_img, y):
    # TODO: Figure 3 - one or two panels:
    #   (a) boxplot or histogram of "ink per image" for each class (black vs grey, hatched bars),
    #   (b) histogram of non-zero pixel intensities (shows most values are 0 or near 255,
    #       i.e. the images are close to binary).
    # HD tip: (a) shows how separable the classes are on a single feature;
    # (b) justifies scaling to [0,1] by dividing by 255 rather than per-pixel standardisation.
    raise NotImplementedError


fig = plot_feature_distributions(X_train_img, y_train_raw)
save_fig(fig, "fig3_feature_distributions")

In [ ]:
def plot_pca_projection(X_img, y):
    # TODO: Figure 4 - fit PCA on the TRAINING images only (flattened, /255) and plot:
    #   (a) scatter of PC1 vs PC2, class 1 = hollow circles, class 7 = filled triangles (B&W friendly),
    #   (b) cumulative explained variance vs number of components (mark where 90% / 95% is reached).
    # HD tip: if the two classes form two clearly separated clouds in just 2 components, the
    # problem is close to linearly separable -> supports a linear first model. Note also how
    # many components hold 95% of variance (far fewer than 784): the data's intrinsic dimension
    # is low, which is why a heavily regularised model works with only 100 samples.
    raise NotImplementedError


fig = plot_pca_projection(X_train_img, y_train_raw)
save_fig(fig, "fig4_pca_projection")

### 1.3 Preprocessing
Every step below must be **justified in the report by a finding from 1.1–1.2**.

| Step | Justification (fill in) |
|---|---|
| Flatten 28×28 → 784 | Classical classifiers expect a feature vector |
| Scale to [0, 1] (÷255) | Fig 3: near-binary values; keeps sparsity; puts all pixels on one scale for regularisation / RBF distances |
| Encode labels 7 → 1 (positive), 1 → 0 | Makes precision/recall/F1 refer to a named class; state which class is positive |
| (Optional) Drop zero-variance pixels | Table 1.1: N pixels are always 0 — fit inside the pipeline so it only sees CV-training folds |
| No augmentation / no test-set fitting | Keeps the evaluation honest |

In [ ]:
def flatten_and_scale(X_img):
    # TODO: reshape (n, 28, 28) -> (n, 784) and convert to float in [0, 1] by dividing by 255.
    # This transform is stateless (it learns nothing from the data), so applying it to the
    # test set later cannot leak information.
    raise NotImplementedError


def encode_labels(y_raw, positive_digit=7):
    # TODO: return a 0/1 integer array where `positive_digit` -> 1 and the other digit -> 0.
    # State in the report why you chose 7 as the positive class (any choice is fine if justified).
    raise NotImplementedError


X_train = flatten_and_scale(X_train_img)
y_train = encode_labels(y_train_raw)
print(X_train.shape, X_train.min(), X_train.max(), np.bincount(y_train))

### 1.4 Findings (write in report)
Summarise in 4–6 bullet points. Template:
- **Size/balance:** … → implication for metric choice.
- **Sparsity / unused pixels:** … → implication for preprocessing and for d ≫ n.
- **Class differences (ink, width, mean images):** … → implication: classes are easy/hard to separate.
- **Within-class variety / hard cases:** … → where errors are likely.
- **PCA:** … → evidence of near-linear separability and low intrinsic dimension → motivates regularised linear model first.

---
# Task 2 — Build a Classification System (25%)

**Model 1: L2-regularised Logistic Regression** (you may swap in a linear SVM, but keep the justification data-driven).

**Data-driven justification to write up (HD needs this link):**
- *n = 100 ≪ d = 784* → high risk of overfitting; a linear model with **regularisation (C)** has low variance.
- Fig 4 (PCA) shows near-linear separability → a linear decision boundary is sufficient.
- Fig 2(c) difference image → the model's weight vector can be **reshaped to 28×28 and inspected**, making the model interpretable ("special design that connects data to the model").
- Produces calibrated probabilities → enables ROC-AUC and log-loss, not just hard labels.
- Convex optimisation → same answer every run (reproducible).

**Metrics (justify each):**
| Metric | Why |
|---|---|
| Accuracy (primary) | Classes are balanced (Task 1) and both error types cost the same → accuracy is unbiased and easy to interpret |
| F1 (positive = 7) | Sensitive to errors on one class; guards against a model that ignores a class |
| ROC-AUC | Threshold-independent; ranks models even when accuracy ties at 100% on small folds |
| Log-loss | Finest-grained; rewards confident correct predictions — useful tie-breaker with n=100 |
| Confusion matrix | Shows *which* digit gets confused for which |

Mention balanced accuracy as the alternative you would use if the classes were imbalanced.

In [ ]:
# [provided] Scoring dictionary used by every cross-validation call in Tasks 2 and 3.
# sklearn maximises scores, so log-loss appears as "neg_log_loss" (closer to 0 is better).
SCORING = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "neg_log_loss": "neg_log_loss",
}
PRIMARY_METRIC = "accuracy"   # TODO: change if you justify a different primary metric


# [provided] Cross-validation scheme, shared by both models so the comparison is fair.
# With only 100 samples, a single 5-fold split has high variance (each fold = 20 images,
# one error = 5% accuracy). Repeating 5-fold CV 10 times with different shuffles averages
# out that noise -> 50 train/validation splits per hyperparameter setting.
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

In [ ]:
def build_logreg_pipeline(C=1.0, penalty="l2"):
    # TODO: return an sklearn Pipeline with these steps:
    #   ("variance", VarianceThreshold(0.0))  # optional: drops always-zero pixels, fit per fold
    #   ("clf", LogisticRegression(C=C, penalty=penalty, solver="liblinear" or "saga",
    #                              max_iter=5000, random_state=RANDOM_STATE))
    # Why a Pipeline: any step that LEARNS from data (variance filter, scaler, PCA) is re-fitted
    # inside each CV training fold, so the validation fold never leaks into training.
    # Note: input is already /255, so a StandardScaler is optional — if you add one, justify it
    # (and note it inflates near-constant border pixels).
    raise NotImplementedError

In [ ]:
def evaluate_with_cv(model, X, y, cv=CV, scoring=SCORING):
    # TODO: run cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=True)
    # and return a tidy DataFrame: rows = metrics, columns = train mean, train std, val mean, val std.
    # HD tip: comparing train vs validation scores shows over/under-fitting directly.
    raise NotImplementedError


logreg_default = build_logreg_pipeline(C=1.0)
baseline_cv = evaluate_with_cv(logreg_default, X_train, y_train)
baseline_cv

In [ ]:
# TODO: Fit the default Logistic Regression on the FULL training set (100 images).
#   logreg_default.fit(X_train, y_train)
# Then report training-set accuracy and the confusion matrix ON TRAINING DATA ONLY
# (label it clearly as training performance — it is optimistic).
raise NotImplementedError

In [ ]:
def plot_logreg_weights(pipeline):
    # TODO: Figure 5 - reshape the learned coefficient vector back to 28x28 and show it with a
    # symmetric diverging greyscale map + colorbar.
    #   - If you used VarianceThreshold, put the coefficients back into a zero 784-vector at
    #     pipeline["variance"].get_support() positions before reshaping.
    # HD tip: compare with Fig 2(c). Dark (positive) weights where 7s have ink (top bar),
    # light (negative) weights along the vertical centre stroke of 1s. This shows the model
    # learned the same structure you found in the data — strong evidence for the model choice.
    raise NotImplementedError


fig = plot_logreg_weights(logreg_default)
save_fig(fig, "fig5_logreg_weights")

---
# Task 3 — Comparison of Two Classifiers & Hyperparameter Tuning (35%)

**Model 2: Support Vector Machine with RBF kernel** (non-linear, kernel-based, margin-maximising).

**Key differences to explain in the report and link to results:**
| | Logistic Regression | RBF SVM |
|---|---|---|
| Decision boundary | Linear hyperplane | Non-linear, any shape (via kernel) |
| Loss | Log-loss (probabilistic) | Hinge loss (margin) |
| Output | Calibrated probabilities | Decision score (probabilities only via Platt scaling) |
| Capacity control | C | C **and** γ (kernel width) |
| Interpretability | Weight image (Fig 5) | Support vectors only |
| Risk with n=100 | Underfit if data non-linear | Overfit if γ large / C large |

**Expectation to test:** if Task 1 showed near-linear separability, the extra flexibility of the SVM should give *little or no* gain — and may overfit. Your results should confirm or refute this.

**Hyperparameters (rubric HD: 2–3 parameters, comprehensive range):**
- Logistic Regression: `C` ∈ logspace(-4, 4, 17), `penalty` ∈ {l1, l2}
- RBF SVM: `C` ∈ logspace(-2, 4, 13), `gamma` ∈ logspace(-5, 0, 11) (plus `"scale"`)
- (Optional 3rd parameter for either: PCA `n_components` ∈ {10, 20, 50, None})

### 3.1 Validation approach (write in report)
- **Repeated stratified 5-fold CV (5 × 10 repeats)** on the 100 training images only. Stratified keeps the 1/7 ratio in every fold.
- Why not a hold-out split: 100 images → a 20-image validation set gives ±5% steps and very noisy estimates.
- Why repeats: reduces variance due to the particular split.
- **Selection rule:** highest mean validation `PRIMARY_METRIC`; break ties with mean log-loss, then prefer the **simpler model** (smaller C / smaller γ). Optionally apply the **one-standard-error rule**.
- **Optional (strong HD):** nested CV — an outer 5-fold loop around `GridSearchCV` gives an *unbiased* estimate of the tuned model's performance, because the inner selection otherwise slightly overfits the CV score.

In [ ]:
def build_svm_pipeline(C=1.0, gamma="scale"):
    # TODO: return Pipeline([("variance", VarianceThreshold(0.0)),
    #                        ("clf", SVC(kernel="rbf", C=C, gamma=gamma,
    #                                    probability=True, random_state=RANDOM_STATE))])
    # probability=True is needed for ROC-AUC and log-loss (Platt scaling; slower but n is tiny).
    # Note: gamma="scale" = 1 / (n_features * X.var()) -> a sensible data-driven midpoint;
    # make sure your gamma grid brackets this value (print it!).
    raise NotImplementedError

In [ ]:
# TODO: Define the two search grids (keys use the pipeline step prefix "clf__").
LOGREG_GRID = {
    # "clf__C": np.logspace(-4, 4, 17),
    # "clf__penalty": ["l1", "l2"],
}
SVM_GRID = {
    # "clf__C": np.logspace(-2, 4, 13),
    # "clf__gamma": np.logspace(-5, 0, 11),
}
raise NotImplementedError

In [ ]:
def run_grid_search(pipeline, param_grid, X, y, cv=CV, scoring=SCORING, refit=PRIMARY_METRIC):
    # TODO:
    #   1. GridSearchCV(pipeline, param_grid, cv=cv, scoring=scoring, refit=refit,
    #                   return_train_score=True, n_jobs=-1)
    #   2. .fit(X, y) and time it (time.perf_counter()).
    #   3. Convert .cv_results_ into a DataFrame keeping params + mean/std of every train/test metric.
    #   4. Return (fitted_search, results_df).
    # Remember: refit=True retrains the best configuration on ALL 100 training images.
    raise NotImplementedError


logreg_search, logreg_results = run_grid_search(build_logreg_pipeline(), LOGREG_GRID, X_train, y_train)
svm_search, svm_results = run_grid_search(build_svm_pipeline(), SVM_GRID, X_train, y_train)

### 3.2 Effect of hyperparameters — figures

In [ ]:
def plot_validation_curves_logreg(results_df):
    # TODO: Figure 6 - for each penalty (l1 solid, l2 dashed):
    #   x = C on a log axis; y = mean validation accuracy with a shaded +-1 std band;
    #   also plot mean TRAINING accuracy (thin dotted line) to show the generalisation gap.
    #   Add a second panel with validation log-loss vs C (finer-grained than accuracy).
    #   Mark the selected C with a vertical line.
    # HD tip: describe the three regimes: very small C = underfitting (too much regularisation,
    # both curves low); middle = best; large C = train hits 100% while validation flattens/drops.
    raise NotImplementedError


fig = plot_validation_curves_logreg(logreg_results)
save_fig(fig, "fig6_logreg_validation_curves")

In [ ]:
def plot_svm_heatmap(results_df, metric="mean_test_accuracy"):
    # TODO: Figure 7 - heatmap of mean validation accuracy over (gamma rows x C columns).
    #   - Use pivot_table, imshow with cmap="Greys", annotate each cell with the value (2 d.p.),
    #     log-scale tick labels, mark the best cell with a box/asterisk.
    #   - Repeat for log-loss as a second panel if space allows.
    # HD tip: explain the diagonal structure: large gamma -> each support vector only "sees"
    # its own neighbourhood -> memorises training data (overfit); tiny gamma -> kernel is
    # almost linear -> behaves like Model 1; small C -> wide soft margin (underfit).
    raise NotImplementedError


fig = plot_svm_heatmap(svm_results)
save_fig(fig, "fig7_svm_heatmap")

In [ ]:
def plot_svm_slices(results_df, best_C, best_gamma):
    # TODO: Figure 8 - two 1-D validation curves for the SVM (cleaner to read than a heatmap):
    #   (a) accuracy vs gamma at best_C, (b) accuracy vs C at best_gamma, each with train + val
    #   lines and a +-1 std band.
    raise NotImplementedError


# fig = plot_svm_slices(svm_results, best_C=..., best_gamma=...)
# save_fig(fig, "fig8_svm_slices")

In [ ]:
def plot_learning_curves(models, X, y, cv=CV):
    # TODO: Figure 9 - learning_curve(...) for both tuned models with
    #   train_sizes=np.linspace(0.2, 1.0, 6). Plot mean validation accuracy vs number of
    #   training images (model 1 solid/circles, model 2 dashed/squares).
    # HD tip: shows which model is more data-efficient and whether more data would still help —
    # a strong discussion point given the deliberately tiny training set.
    raise NotImplementedError


# fig = plot_learning_curves({"LogReg": logreg_search.best_estimator_,
#                             "RBF SVM": svm_search.best_estimator_}, X_train, y_train)
# save_fig(fig, "fig9_learning_curves")

### 3.3 Selected hyperparameters and CV comparison

In [ ]:
def summarise_best(search, results_df, name):
    # TODO: return a one-row DataFrame: model name, best params, mean +- std CV for every metric,
    # number of configurations tried, fit time.
    # Optional one-standard-error rule: among configs whose mean accuracy is within 1 std error
    # (std / sqrt(n_splits)) of the best, pick the most regularised one. Report if it differs.
    # HD tip: if many configs tie at the same accuracy (likely with n=100), say so and show
    # how log-loss broke the tie — this demonstrates a well-justified selection metric.
    raise NotImplementedError


cv_comparison = pd.concat([
    summarise_best(logreg_search, logreg_results, "Logistic Regression"),
    summarise_best(svm_search, svm_results, "RBF SVM"),
])
cv_comparison

In [ ]:
# OPTIONAL (strong HD): nested cross-validation for an unbiased estimate of each tuned model.
# TODO:
#   outer = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
#   inner = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
#   for each model: scores = cross_validate(GridSearchCV(pipe, grid, cv=inner, scoring=..., refit=PRIMARY_METRIC),
#                                            X_train, y_train, cv=outer, scoring=SCORING)
#   Report outer-fold mean +- std. Compare with the (optimistic) best_score_ from 3.3.
pass

### 3.4 Final evaluation on the test set (run ONCE)
Hyperparameters are now frozen. This is the first and only time the classifiers see the test set.
Do not go back and change anything based on these numbers — if you do, say so honestly in the report.

In [ ]:
# [provided] Prepare the test set with the SAME stateless transforms as training.
X_test = flatten_and_scale(X_test_img)
y_test = encode_labels(y_test_raw)
print(X_test.shape, np.bincount(y_test))

In [ ]:
def evaluate_on_test(models, X, y):
    # TODO: for each final model (search.best_estimator_):
    #   - y_pred = model.predict(X); y_prob = model.predict_proba(X)[:, 1]
    #   - compute accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, log-loss
    #   - 95% bootstrap confidence interval for accuracy (resample test indices 1000x)
    # Return a DataFrame (rows = models, columns = metrics).
    raise NotImplementedError


final_models = {
    "Logistic Regression": None,   # TODO: logreg_search.best_estimator_
    "RBF SVM": None,               # TODO: svm_search.best_estimator_
}
test_results = evaluate_on_test(final_models, X_test, y_test)
test_results

In [ ]:
def plot_test_diagnostics(models, X, y):
    # TODO: Figure 10 - for both models:
    #   (a) confusion matrices side by side (ConfusionMatrixDisplay, cmap="Greys", labels "1","7")
    #   (b) ROC curves on one axis (RocCurveDisplay.from_estimator, different line styles)
    raise NotImplementedError


fig = plot_test_diagnostics(final_models, X_test, y_test)
save_fig(fig, "fig10_test_diagnostics")

In [ ]:
def plot_misclassified(models, X_img, X, y, max_show=12):
    # TODO: Figure 11 - show the test images each model got wrong, titled "true -> predicted (p=0.xx)".
    #   Also print how many errors the two models SHARE vs make uniquely.
    # HD tip: shared errors are usually genuinely ambiguous digits (1s with a long top hook,
    # 7s without a crossbar written almost vertically). Discuss whether the SVM's non-linearity
    # fixed any of the LogReg's errors -> ties back to the model-difference table.
    raise NotImplementedError


fig = plot_misclassified(final_models, X_test_img, X_test, y_test)
save_fig(fig, "fig11_misclassified")

In [ ]:
# OPTIONAL (strong HD): McNemar's test - are the two models' test errors significantly different?
# TODO: build the 2x2 table of (LogReg correct?, SVM correct?) counts; with b, c = disagreement cells:
#   statistic = (|b - c| - 1)^2 / (b + c), p-value from chi-square with 1 dof
#   (scipy.stats.chi2.sf(statistic, 1)), or use statsmodels.stats.contingency_tables.mcnemar.
# If p > 0.05: "the difference is not statistically significant" -> prefer the simpler model.
pass

### 3.5 Discussion (write in report)
Answer these explicitly:
1. Which model won on CV and on test? By how much, and is it significant (CI overlap / McNemar)?
2. Do the results match what the model differences predicted (linear vs non-linear, Task 1 PCA)?
3. Which hyperparameter mattered most? Which regimes over/under-fit (Figs 6–8)?
4. Was the CV estimate close to the test score? (If CV was optimistic, explain: selection bias / small n.)
5. Trade-offs beyond accuracy: interpretability, probability calibration, training time, data efficiency (Fig 9).
6. Final recommendation, with one sentence of justification.

---
# Task 4 — Clustering Analysis of the Test Set (25%)

The assignment requires clustering the **digit-7 images from the test set**. Test labels are used **only** to select the 7s (as instructed); K-means itself never sees any label.

**Plan**
1. K-means with **k = 100**, visualise the 100 centroids and the cluster-size distribution.
2. Find the optimal **k < 100** using several, complementary criteria:
   - Elbow on inertia (within-cluster sum of squares)
   - Silhouette score (higher = better)
   - Calinski–Harabasz (higher = better)
   - Davies–Bouldin (lower = better)
   - *Strong HD extras:* stability across random seeds (mean pairwise ARI), and/or a Ward dendrogram of the 100 centroids from step 1 (links the two steps the brief asks for).
3. Justify one k by weighing the evidence (they rarely all agree — explain the trade-off).
4. Visualise centroids **and** real representative examples per cluster, then interpret each cluster's writing style.

In [ ]:
def select_digit(X_img, y_raw, digit=7):
    # TODO: return the flattened, /255-scaled images of `digit` from the given set.
    # Print how many there are (this n bounds the sensible range of k).
    raise NotImplementedError


X7 = select_digit(X_test_img, y_test_raw, digit=7)
print("Digit-7 test images:", X7.shape)

### 4.1 K-means with k = 100

In [ ]:
def fit_kmeans(X, k, seed=RANDOM_STATE, n_init=10):
    # TODO: fit and return KMeans(n_clusters=k, n_init=n_init, random_state=seed).
    # n_init > 1 restarts from several random seeds and keeps the lowest-inertia solution,
    # reducing the chance of a poor local minimum (K-means is not convex).
    raise NotImplementedError


km100 = fit_kmeans(X7, 100)

In [ ]:
def plot_centroid_grid(centroids, sizes=None, ncols=10, title=""):
    # TODO: Figure 12 - draw each centroid reshaped to 28x28 in a grid (cmap="gray_r", axes off).
    #   If sizes are given, put "n=..." under each image, and sort clusters by size (largest first).
    # Reusable: you will call this again for the optimal k.
    raise NotImplementedError


def plot_cluster_sizes(labels, title=""):
    # TODO: bar chart (black bars) of cluster sizes sorted descending.
    # HD tip: for k=100 with only ~N sevens, expect many tiny clusters (1-3 images) -> the
    # model is memorising individual images rather than finding writing styles. This is the
    # motivation for searching for a smaller k.
    raise NotImplementedError


fig = plot_centroid_grid(km100.cluster_centers_, np.bincount(km100.labels_), title="k = 100")
save_fig(fig, "fig12_k100_centroids")
fig = plot_cluster_sizes(km100.labels_, title="Cluster sizes, k = 100")
save_fig(fig, "fig13_k100_sizes")

### 4.2 Choosing the optimal k (< 100)

In [ ]:
K_RANGE = list(range(2, 31)) + [35, 40, 50, 60, 70, 80, 90, 99]   # dense where it matters, sparse above

def sweep_k(X, k_values, seed=RANDOM_STATE):
    # TODO: for each k, fit K-means and record in a DataFrame:
    #   k, inertia, silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    #   min/median cluster size.
    # Print progress; this takes a minute or two.
    raise NotImplementedError


k_sweep = sweep_k(X7, K_RANGE)
k_sweep.head()

In [ ]:
def plot_k_selection(k_sweep, chosen_k=None):
    # TODO: Figure 14 - a 2x2 panel: inertia (elbow), silhouette, Calinski-Harabasz, Davies-Bouldin
    #   vs k. Use markers + black lines, mark chosen_k with a vertical dashed line on every panel,
    #   and state "higher is better"/"lower is better" in each title.
    # Elbow helper (optional, objective rather than eyeballed): the "kneedle" idea = the k whose
    # point on the normalised inertia curve is furthest from the straight line joining the
    # first and last points.
    raise NotImplementedError


fig = plot_k_selection(k_sweep)
save_fig(fig, "fig14_k_selection")

In [ ]:
def stability_by_k(X, k_values, n_seeds=10):
    # OPTIONAL (strong HD): for each k, fit K-means with n_seeds different random_state values
    # (n_init=1) and compute the mean pairwise adjusted_rand_score between the label sets.
    # A "real" structure is found again and again (ARI near 1); an arbitrary partition is not.
    # Plot mean ARI vs k (Figure 15).
    raise NotImplementedError


def dendrogram_of_k100_centroids(km100):
    # OPTIONAL (strong HD): hierarchical (Ward) clustering of the 100 centroids, weighted by size.
    #   Z = linkage(km100.cluster_centers_, method="ward"); dendrogram(Z, no_labels=True, color_threshold=0)
    # Large vertical gaps between merge heights suggest a natural number of groups.
    # This uses the k=100 solution directly to answer "what is the optimal number < 100",
    # which links the two clustering steps in the brief.
    raise NotImplementedError

**Write-up guidance for the choice of k**
- Report a table of the metrics at the candidate k values (e.g. the elbow point, silhouette peak, DB minimum).
- Note known caveats: silhouette in 784-D Euclidean space is often low and tends to favour very small k; inertia always decreases with k so you need the *elbow*, not the minimum.
- Pick the k where several criteria roughly agree **and** the centroids are visually distinct and interpretable (no two centroids that look identical, no clusters of 1–2 images).
- State the final k clearly with a 2–3 sentence data-driven justification.

In [ ]:
OPTIMAL_K = None   # TODO: set your chosen k (< 100) based on the evidence above
km_opt = fit_kmeans(X7, OPTIMAL_K)

### 4.3 Visualising and interpreting the optimal clusters

In [ ]:
def plot_cluster_representatives(X, km, n_examples=8):
    # TODO: Figure 16 - one ROW per cluster:
    #   column 0 = the centroid (labelled "centroid, n=..."),
    #   columns 1..n_examples = the real images CLOSEST to that centroid
    #   (hint: distances = np.linalg.norm(X[members] - center, axis=1); argsort).
    #   Optionally add the single FURTHEST member at the end, labelled "outlier".
    # Why both: a centroid is an average and can look blurry or unlike any real digit;
    # the nearest real examples (medoid-like) show what the cluster actually contains.
    raise NotImplementedError


fig = plot_centroid_grid(km_opt.cluster_centers_, np.bincount(km_opt.labels_),
                         ncols=min(OPTIMAL_K, 10), title=f"k = {OPTIMAL_K}")
save_fig(fig, "fig15_optimal_centroids")
fig = plot_cluster_representatives(X7, km_opt)
save_fig(fig, "fig16_cluster_representatives")

In [ ]:
def plot_clusters_2d(X, labels, centroids):
    # TODO: Figure 17 - project X7 to 2D with PCA (fit on X7), scatter each cluster with a
    # different MARKER (B&W friendly), and overlay the projected centroids as large black crosses.
    # Optional: small thumbnails of the centroids at their 2D location (matplotlib OffsetImage).
    # Comment on which clusters overlap and which are well separated.
    raise NotImplementedError


fig = plot_clusters_2d(X7, km_opt.labels_, km_opt.cluster_centers_)
save_fig(fig, "fig17_clusters_pca")

In [ ]:
def describe_clusters(X, km):
    # TODO: return a DataFrame, one row per cluster: size, % of 7s, mean ink, mean stroke width,
    # mean slant (optional: angle of the principal axis of the ink pixels), mean distance to centroid
    # (tightness). Then add a free-text column "Writing style" that YOU fill in, e.g.
    #   "upright 7 with a horizontal crossbar", "strongly right-slanted, thin stroke",
    #   "European 7 with long curved top", "thick/bold stroke".
    # HD tip: the rubric asks for "in-depth analysis of the key patterns revealed by each cluster"
    # -> this table + Figure 16 is exactly that.
    raise NotImplementedError


cluster_table = describe_clusters(X7, km_opt)
cluster_table

### 4.4 Findings (write in report)
- How K-means was run (initialisation `k-means++`, `n_init`, seed, features = scaled pixels).
- What k = 100 looked like and why it is too many.
- How the optimal k was chosen (criteria + figures) and the final k.
- What each cluster represents (styles: crossbar vs none, slant, stroke thickness, top-bar shape), and any outliers.
- Limitation: K-means assumes roughly spherical clusters of similar size in Euclidean pixel space; small shifts of a digit change many pixels, so clusters partly capture *position/slant* rather than *style*.

---
## Appendix — Reproducibility checklist
- [ ] `RANDOM_STATE` used for every split, model and K-means run
- [ ] *Restart & Run All* completes with no errors
- [ ] All figures saved to `figures/` and inserted into the report
- [ ] Test set used only in 3.4 and Task 4